In [ ]:
import json
import os
from datetime import datetime

class Barang:
    def __init__(self, id_barang, nama, harga_jual, harga_pokok, pin_admin):
        self.id_barang = id_barang
        self.nama = nama
        self.harga_jual = harga_jual
        self.__harga_pokok = harga_pokok
        self.__pin_admin = pin_admin

    def info(self):
      margin = self.harga_jual - self.__harga_pokok
      return f"{self.id_barang:<5} | Produk: {self.nama:<15} | {self.harga_jual:>12} | {margin:>11} |"

    def to_dict(self):
        return {
            "tipe": "umum",
            "id": self.id_barang,
            "nama": self.nama,
            "harga_jual": self.harga_jual,
            "harga_pokok": self.__harga_pokok
        }

    def get_harga_pokok(self):
        return self.__harga_pokok

    def update_harga_pokok(self, pin, harga_baru):
        if pin == self.__pin_admin:
            if harga_baru > 0:
                self.__harga_pokok = harga_baru
                print(f"✅ Harga pokok {self.nama} berhasil diperbarui.")
            else:
                print("❌ Gagal: Harga pokok tidak boleh nol atau negatif!")
        else:
            print("❌ Gagal: PIN salah! Anda tidak memiliki akses.")

class BarangElektronik(Barang):
    def __init__(self,id_barang, nama, harga_jual, harga_pokok, pin_admin, garansi):
        super().__init__(id_barang, nama, harga_jual, harga_pokok, pin_admin)
        self.garansi = garansi

    def info(self):
        return f"{super().info()} Garansi: {self.garansi} bln"

    def update_harga_pokok(self, pin, harga_baru):
        if pin == self._Barang__pin_admin:
            if harga_baru > 0:
                self._Barang__harga_pokok = harga_baru
                print(f"✅ Harga pokok {self.nama} berhasil diperbarui (Elektronik).")
            else:
                print("❌ Gagal: Harga pokok tidak boleh nol atau negatif!")
        else:
            print("❌ Gagal: PIN salah! Anda tidak memiliki akses.")

    def to_dict(self):
        d = super().to_dict()
        d.update({"tipe": "elektronik", "garansi": self.garansi})
        return d


class BarangKonsumsi(Barang):
    def __init__(self, id_barang, nama, harga_jual, harga_pokok, pin_admin, tgl_exp):
        super().__init__(id_barang, nama, harga_jual, harga_pokok, pin_admin)
        self.tgl_exp = tgl_exp

    def info(self):
        try:
            tgl_exp_date = datetime.strptime(self.tgl_exp, "%Y-%m-%d").date()
            today = datetime.now().date()

            status = "KADALUARSA" if tgl_exp_date < today else "AMAN"
        except:
            status = "FORMAT SALAH"

        return f"{super().info()} Exp: {self.tgl_exp} ({status})"

    def update_harga_pokok(self, pin, harga_baru):
        if pin == self._Barang__pin_admin:
            if harga_baru > 0:
                self._Barang__harga_pokok = harga_baru
                print(f"✅ Harga pokok {self.nama} berhasil diperbarui (Konsumsi).")
            else:
                print("❌ Gagal: Harga pokok tidak boleh nol atau negatif!")
        else:
            print("❌ Gagal: PIN salah! Anda tidak memiliki akses.")

    def to_dict(self):
        d = super().to_dict()
        d.update({"tipe": "konsumsi", "tgl_exp": self.tgl_exp})
        return d


class GudangPolimorfik:
    def __init__(self, file_db='gudang_v3.json', reset=False):
        self.file_db = file_db
        self.koleksi = []
        if not reset:
            self.muat_data()

    def tambah_barang(self, objek_barang):
        self.koleksi.append(objek_barang)
        self.simpan_data()

    def simpan_data(self):
        with open(self.file_db, 'w') as f:
            data = [b.to_dict() for b in self.koleksi]
            json.dump(data, f, indent=4)

    def muat_data(self):
        if os.path.exists(self.file_db):
            with open(self.file_db, 'r') as f:
                data_list = json.load(f)
                self.koleksi = []

                for d in data_list:
                    if d['tipe'] == "elektronik":
                        obj = BarangElektronik(
                            d['id'], d['nama'], d['harga_jual'],
                            d.get('harga_pokok', 0), "1234",
                            d['garansi']
                        )
                    elif d['tipe'] == "konsumsi":
                        obj = BarangKonsumsi(
                            d['id'], d['nama'], d['harga_jual'],
                            d.get('harga_pokok', 0), "1234",
                            d['tgl_exp']
                        )
                    else:
                        obj = Barang(
                            d['id'], d['nama'], d['harga_jual'],
                            d.get('harga_pokok', 0), "1234"
                        )

                    self.koleksi.append(obj)

    def laporan_stok(self):
        print("\n" + "=" * 95)
        print(f"{  'ID':5} | {'NAMA BARANG':23} | {'HARGA JUAL':12} | {'MARGIN':11} | KETERANGAN")
        print("-" * 95)

        for b in self.koleksi:
            print(b.info())

        print("-" * 95)


def main():
    app = GudangPolimorfik(reset=True)

    if not app.koleksi:
        print("Membuat data awal...\n")

        barang1 = BarangElektronik("E01","Laptop Pro", 12000000, 10000000, "1234", 12)
        barang2 = BarangKonsumsi("K01","Roti Coklat", 15000, 10000, "1234", "2024-05-10")
        barang3 = Barang("B01","Kabel Data", 25000, 15000, "1234")
        barang4 = BarangKonsumsi("K02","Cokolatos", 10000, 8000, "1234", "2027-02-11")

        app.tambah_barang(barang1)
        app.tambah_barang(barang2)
        app.tambah_barang(barang3)
        app.tambah_barang(barang4)

    print("\n=== DATA AWAL ===")
    app.laporan_stok()

    print("\n=== DEMO ENKAPSULASI ===")

    for barang in app.koleksi:
        print(f"\nBarang: {barang.nama}")
        print("Harga pokok awal:", barang.get_harga_pokok())

        print("Coba ubah (PIN salah):")
        barang.update_harga_pokok("0000", barang.get_harga_pokok() - 1000)

        print("Coba ubah (PIN benar):")
        barang.update_harga_pokok("1234", barang.get_harga_pokok() - 1000)

        print("Harga pokok sekarang:", barang.get_harga_pokok())

    print("\n=== DATA SETELAH ENKAPSULASI ===")
    app.laporan_stok()

    print("\n(Update harga pokok akan mempengaruhi margin)")
if __name__ == "__main__":
    main()

Membuat data awal...


=== DATA AWAL ===

ID    | NAMA BARANG             | HARGA JUAL   | MARGIN      | KETERANGAN
-----------------------------------------------------------------------------------------------
E01   | Produk: Laptop Pro      |     12000000 |     2000000 | Garansi: 12 bln
K01   | Produk: Roti Coklat     |        15000 |        5000 | Exp: 2024-05-10 (KADALUARSA)
B01   | Produk: Kabel Data      |        25000 |       10000 |
K02   | Produk: Cokolatos       |        10000 |        2000 | Exp: 2027-02-11 (AMAN)
-----------------------------------------------------------------------------------------------

=== DEMO ENKAPSULASI ===

Barang: Laptop Pro
Harga pokok awal: 10000000
Coba ubah (PIN salah):
❌ Gagal: PIN salah! Anda tidak memiliki akses.
Coba ubah (PIN benar):
✅ Harga pokok Laptop Pro berhasil diperbarui (Elektronik).
Harga pokok sekarang: 9999000

Barang: Roti Coklat
Harga pokok awal: 10000
Coba ubah (PIN salah):
❌ Gagal: PIN salah! Anda tidak memiliki akses.
Co